# 12-lead ECG ablation study (PTB-XL)

This notebook evaluates how lead subsets, sampling rate, and model family affect multi-label diagnostic-superclass prediction on PTB-XL. It is designed for a resource-conscious baseline: a compact separable CNN using the limb leads I, II, III, aVR, aVL, and aVF.

> **Important:** Execute this notebook on the licensed/local PTB-XL files before treating metrics as experimental results.

## Experimental protocol

- Dataset: PTB-XL records at 100 Hz (`filename_lr`) with diagnostic superclass labels (`NORM`, `MI`, `STTC`, `CD`, `HYP`).
- Split: official `strat_fold == 10` test set; a validation split is drawn only from training records.
- Metric: macro AUROC is primary; macro F1 and per-class AUROC check balance across ailments.
- Ablations: 12 vs 6 vs 3 leads, 100 vs 50 Hz, and compact CNN vs linear/RF feature baselines.
- Reproducibility: fixed seed, no test-set threshold tuning, model size and latency recorded.

In [ ]:
# pip install wfdb tensorflow scikit-learn pandas numpy
from pathlib import Path
import ast, json, random, time
import numpy as np
import pandas as pd
import wfdb
import tensorflow as tf
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
DATA_ROOT = Path('dataset')  # directory containing ptbxl_database.csv and records100/
FS = 100
CLASS_ORDER = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
LEAD_SETS = {
    '12_lead': list(range(12)),
    'limb_6': [0, 1, 2, 3, 4, 5],
    'basic_3': [0, 1, 2],
    'lead_I': [0],  # Lead I = LA - RA; RL is the reference/ground electrode.
    'precordial_6': [6, 7, 8, 9, 10, 11],
}


In [ ]:
# Read official metadata and map SCP statements to diagnostic superclasses.
meta = pd.read_csv(DATA_ROOT / 'ptbxl_database.csv', index_col='ecg_id')
meta['scp_codes'] = meta['scp_codes'].map(ast.literal_eval)
statements = pd.read_csv(DATA_ROOT / 'scp_statements.csv', index_col=0)
diagnostic = statements[statements.diagnostic == 1]

def diagnostic_superclasses(code_dict):
    return sorted({diagnostic.loc[c, 'diagnostic_class']
                   for c in code_dict if c in diagnostic.index
                   and diagnostic.loc[c, 'diagnostic_class'] in CLASS_ORDER})

meta['labels'] = meta['scp_codes'].map(diagnostic_superclasses)
meta = meta[meta.labels.map(len) > 0].copy()
mlb = MultiLabelBinarizer(classes=CLASS_ORDER)
y_all = mlb.fit_transform(meta.labels)
train_mask = meta.strat_fold != 10
test_mask = ~train_mask
print(f'Records: {len(meta):,}; train: {train_mask.sum():,}; test: {test_mask.sum():,}')
print(pd.Series(y_all.sum(axis=0), index=CLASS_ORDER, name='label_count'))


In [ ]:
# Load records lazily. PTB-XL filename_lr is sampled at 100 Hz and retains all 12 ordered leads.
def load_signals(frame, lead_indices, max_records=None):
    rows = frame.iloc[:max_records] if max_records else frame
    signals = []
    for rel_path in rows.filename_lr:
        sig, fields = wfdb.rdsamp(str(DATA_ROOT / rel_path))
        assert sig.shape[1] == 12, f'Expected 12 leads, got {sig.shape[1]}'
        signals.append(sig[:, lead_indices].astype('float32'))
    return np.stack(signals)

def downsample(x, original_hz, target_hz):
    if target_hz == original_hz:
        return x
    factor = original_hz // target_hz
    if original_hz % target_hz:
        raise ValueError('Use an integer downsampling factor for this baseline.')
    return x[:, ::factor, :]

def normalize_per_record(x):
    mu = x.mean(axis=1, keepdims=True)
    sd = x.std(axis=1, keepdims=True) + 1e-6
    return (x - mu) / sd


In [ ]:
def compact_ecg_cnn(n_samples, n_leads, n_classes=5):
    inp = tf.keras.Input((n_samples, n_leads), name='ecg')
    x = tf.keras.layers.SeparableConv1D(16, 9, strides=2, padding='same', activation='relu')(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.SeparableConv1D(32, 7, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.SeparableConv1D(48, 5, strides=2, padding='same', activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    out = tf.keras.layers.Dense(n_classes, activation='sigmoid', name='diagnoses')(x)
    model = tf.keras.Model(inp, out, name='compact_separable_ecg_cnn')
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=[tf.keras.metrics.AUC(curve='ROC', multi_label=True, name='macro_auc')])
    return model

def waveform_features(x):
    # Cheap interpretable summary for non-neural baselines: mean, SD, extrema, energy per lead.
    return np.concatenate([x.mean(1), x.std(1), x.min(1), x.max(1), (x ** 2).mean(1)], axis=1)

def evaluate(y_true, probabilities, threshold=0.5):
    per_class = roc_auc_score(y_true, probabilities, average=None)
    return {
        'macro_auroc': roc_auc_score(y_true, probabilities, average='macro'),
        'macro_f1': f1_score(y_true, probabilities >= threshold, average='macro', zero_division=0),
        **{f'auroc_{name}': score for name, score in zip(CLASS_ORDER, per_class)}
    }


In [ ]:
# Run one configuration. Start with max_records=500 as a smoke test, then set None for the full study.
def run_cnn_ablation(lead_set='limb_6', target_hz=100, max_records=None, epochs=20):
    indices = LEAD_SETS[lead_set]
    train_frame, test_frame = meta.loc[train_mask], meta.loc[test_mask]
    x_train = normalize_per_record(downsample(load_signals(train_frame, indices, max_records), FS, target_hz))
    x_test = normalize_per_record(downsample(load_signals(test_frame, indices, max_records), FS, target_hz))
    y_train = y_all[train_mask][:len(x_train)]
    y_test = y_all[test_mask][:len(x_test)]
    model = compact_ecg_cnn(x_train.shape[1], x_train.shape[2])
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_macro_auc', mode='max', patience=4, restore_best_weights=True)]
    model.fit(x_train, y_train, validation_split=.1, epochs=epochs, batch_size=64, callbacks=callbacks, verbose=2)
    start = time.perf_counter(); p = model.predict(x_test, batch_size=128, verbose=0); seconds = time.perf_counter() - start
    result = evaluate(y_test, p)
    result.update({'model': 'Compact separable CNN', 'lead_set': lead_set, 'leads': len(indices),
                   'sampling_hz': target_hz, 'parameters': model.count_params(),
                   'ms_per_record': 1000 * seconds / len(x_test)})
    return result, model

# Example full-resolution run:
# limb6_result, limb6_model = run_cnn_ablation('limb_6', 100, max_records=None, epochs=20)
# pd.DataFrame([limb6_result]).round(4)


In [ ]:
# Feature baselines and experiment registry. Do not combine outputs across different subsets/splits.
def run_feature_baseline(lead_set='limb_6', estimator='logistic', max_records=None):
    idx = LEAD_SETS[lead_set]
    xtr = normalize_per_record(load_signals(meta.loc[train_mask], idx, max_records))
    xte = normalize_per_record(load_signals(meta.loc[test_mask], idx, max_records))
    ytr, yte = y_all[train_mask][:len(xtr)], y_all[test_mask][:len(xte)]
    base = (LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5) if estimator == 'logistic'
            else RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED))
    clf = make_pipeline(StandardScaler(), OneVsRestClassifier(base)) if estimator == 'logistic' else OneVsRestClassifier(base)
    clf.fit(waveform_features(xtr), ytr)
    result = evaluate(yte, clf.predict_proba(waveform_features(xte)))
    result.update({'model': estimator, 'lead_set': lead_set, 'leads': len(idx), 'sampling_hz': 100})
    return result

# Suggested registry; execute each row and save only real measured values.
plan = pd.DataFrame([
    ['Compact separable CNN', '12_lead', 100], ['Compact separable CNN', 'limb_6', 100],
    ['Compact separable CNN', 'limb_6', 50], ['Compact separable CNN', 'basic_3', 100],
    ['Logistic regression', 'limb_6', 100], ['Random forest', 'limb_6', 100],
], columns=['model', 'lead_set', 'sampling_hz'])
display(plan)
# results.to_csv('measured_12lead_ablation_results.csv', index=False)


## Decision rule

The final selected deployment configuration is `lead_I` at 100 Hz: a single Lead I waveform (`LA - RA`) acquired with RA, LA, and RL electrodes. Run it alongside the 12-, 6-, and 3-lead comparators, confirm memory/latency on target hardware, and report ailment-level performance. This is a triage/research model only; it must not be used as a clinical diagnostic device without external validation, calibration, and governance.